# Lab 4 — Genetic Tests and Principal Component Analysis
**Sociogenomics 2025/2026 · University of Bologna · Prof. Nicola Barban**

This notebook covers the R analysis component of Lab 4.  
All PLINK commands are run separately in **[Google Cloud Shell](https://shell.cloud.google.com/)**.

---

The data files are loaded automatically from the course GitHub repository. Just run the cells in order.

In [ ]:
# Download pre-computed PLINK results from the course repository
url <- "https://github.com/nicolabarban/sociogenomics_2025_2026/raw/gh_pages/labs/week4/lab4_results.zip"
download.file(url, destfile = "lab4_results.zip", mode = "wb")
unzip("lab4_results.zip", overwrite = TRUE)
cat("Files loaded:\n")
print(list.files(pattern = "hapmap|bmi|1kg|pca"))

In [ ]:
# Install any missing packages (only needed the first time)
if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2")
library(ggplot2)
theme_set(theme_bw(base_size = 13))
cat("Ready.\n")

---
## Part I — Exploring Genetic Data

PLINK exports genotype quality statistics as plain text tables. We load and visualise them here.

### 1.1 SNP and individual missingness

In [ ]:
lmiss <- read.table("hapmap3_summary.lmiss", header = TRUE)
imiss <- read.table("hapmap3_summary.imiss", header = TRUE)

cat("SNPs in dataset:        ", nrow(lmiss), "\n")
cat("Individuals in dataset: ", nrow(imiss), "\n")
cat("SNP missingness range:  ", range(lmiss$F_MISS), "\n")
cat("Ind missingness range:  ", range(imiss$F_MISS), "\n")

ggplot(lmiss, aes(x = F_MISS)) +
  geom_histogram(bins = 50, fill = "steelblue", colour = "white") +
  xlab("Per-SNP missing rate") + ylab("Number of SNPs") +
  ggtitle("SNP missingness")

ggplot(imiss, aes(x = F_MISS)) +
  geom_histogram(bins = 40, fill = "coral", colour = "white") +
  xlab("Per-individual missing rate") + ylab("Number of individuals") +
  ggtitle("Individual missingness")

### 1.2 Minor allele frequency (MAF) distribution

In [ ]:
frq <- read.table("hapmap3_summary.frq", header = TRUE)
cat("Mean MAF:  ", round(mean(frq$MAF), 4), "\n")
cat("Median MAF:", round(median(frq$MAF), 4), "\n")

ggplot(frq, aes(x = MAF)) +
  geom_histogram(bins = 50, fill = "darkgreen", colour = "white") +
  geom_vline(xintercept = 0.05, colour = "red",
             linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = 0.07, y = Inf, vjust = 2,
           label = "MAF = 0.05", colour = "red", size = 4) +
  xlab("Minor allele frequency") + ylab("Number of SNPs") +
  ggtitle("MAF distribution across all SNPs")

In [ ]:
frq$bin <- cut(frq$MAF,
               breaks = c(0, 0.01, 0.05, 0.10, 0.20, 0.51),
               labels = c("< 0.01", "0.01-0.05", "0.05-0.10", "0.10-0.20", ">= 0.20"),
               right  = FALSE)
print(table(frq$bin))

### 1.3 Hardy-Weinberg Equilibrium (HWE) distribution

SNPs deviating from HWE may indicate genotyping errors. The standard QC threshold is $p < 10^{-6}$.

In [ ]:
hwe     <- read.table("hapmap3_summary.hwe", header = TRUE)
hwe_all <- hwe[hwe$TEST == "ALL", ]
hwe_all$log10p <- -log10(hwe_all$P)

ggplot(hwe_all, aes(x = log10p)) +
  geom_histogram(bins = 60, fill = "purple", colour = "white") +
  geom_vline(xintercept = 6, colour = "red",
             linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = 6.3, y = Inf, vjust = 2,
           label = "p = 1e-6", colour = "red", size = 4) +
  xlab(expression(-log[10](p))) + ylab("Number of SNPs") +
  ggtitle("Distribution of HWE test statistics")

In [ ]:
cat("SNPs failing HWE (p < 1e-6):", sum(hwe_all$P < 1e-6, na.rm = TRUE), "\n")
cat("\nTop 10 most deviant SNPs:\n")
print(head(hwe_all[order(hwe_all$P), c("SNP", "CHR", "O.HET.", "E.HET.", "P")], 10))

### 1.4 Per-individual inbreeding coefficient

The inbreeding coefficient $F$ measures deviation from expected heterozygosity:
- $F < -0.15$: excess heterozygosity → possible sample contamination  
- $F > +0.15$: deficit of heterozygosity → possible inbreeding or genotyping error

In [ ]:
het <- read.table("hapmap3_het.het", header = TRUE)

ggplot(het, aes(x = F)) +
  geom_histogram(bins = 50, fill = "orange", colour = "white") +
  geom_vline(xintercept = c(-0.15, 0.15), colour = "red",
             linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = -0.17, y = Inf, vjust = 2, hjust = 1,
           label = "-0.15", colour = "red", size = 4) +
  annotate("text", x =  0.17, y = Inf, vjust = 2, hjust = 0,
           label = "+0.15", colour = "red", size = 4) +
  xlab("Inbreeding coefficient F") + ylab("Number of individuals") +
  ggtitle("Per-individual inbreeding coefficient")

In [ ]:
outliers <- het[het$F < -0.15 | het$F > 0.15, ]
cat("Heterozygosity outliers:", nrow(outliers), "\n")
if (nrow(outliers) > 0) print(outliers[, c("FID", "IID", "F")])

---
## Part II — Principal Component Analysis (PCA)

PCA summarises genome-wide allele frequency variation into orthogonal axes. Individuals with similar ancestry cluster together in PC space.

> PCA was computed in Cloud Shell with PLINK (`--pca 20` on the LD-pruned SNP set). We load the results here.

### 2.1 Scree plot — variance explained by each PC

In [ ]:
eigenval      <- read.table("hapmap3_pca.eigenval", header = FALSE)
colnames(eigenval) <- "eigenvalue"
eigenval$PC   <- seq_len(nrow(eigenval))
eigenval$pct  <- eigenval$eigenvalue / sum(eigenval$eigenvalue) * 100

ggplot(eigenval, aes(x = PC, y = pct)) +
  geom_col(fill = "steelblue", colour = "white") +
  geom_line(aes(group = 1)) +
  geom_point(size = 2) +
  scale_x_continuous(breaks = 1:20) +
  xlab("Principal Component") + ylab("Variance explained (%)") +
  ggtitle("Scree plot")

In [ ]:
print(eigenval[, c("PC", "pct")])

### 2.2 Load PCA scores and population labels

In [ ]:
pc_cols <- c("FID", "IID", paste0("PC", 1:20))
pca     <- read.table("hapmap3_pca.eigenvec", header = FALSE, col.names = pc_cols)

# Note: read.table converts spaces in column names to dots
# e.g. "Sample name" -> "Sample.name", "Superpopulation name" -> "Superpopulation.name"
geo <- read.table("1kg_samples.txt", sep = "\t", header = TRUE)

data <- merge(pca, geo[, c("Sample.name", "Population.code", "Population.name",
                             "Superpopulation.code", "Superpopulation.name")],
              by.x = "IID", by.y = "Sample.name")

cat("Individuals with population labels:", nrow(data), "\n")
print(table(data$Superpopulation.name))

### 2.3 PC1 vs PC2 — coloured by superpopulation

You should see five clearly separated clusters:  
**AFR** (African), **EUR** (European), **EAS** (East Asian), **SAS** (South Asian), **AMR** (Admixed American).

In [ ]:
ggplot(data, aes(x = PC1, y = PC2, colour = Superpopulation.name)) +
  geom_point(alpha = 0.7, size = 1.5) +
  xlab("PC1") + ylab("PC2") +
  labs(colour = "Superpopulation",
       title  = "PCA — continental ancestry (PC1 vs PC2)")

### 2.4 PC1 vs PC2 — coloured by sub-population

In [ ]:
ggplot(data, aes(x = PC1, y = PC2, colour = Population.name)) +
  geom_point(alpha = 0.7, size = 1.5) +
  xlab("PC1") + ylab("PC2") +
  labs(colour = "Population",
       title  = "PCA — sub-population (PC1 vs PC2)") +
  theme(legend.text = element_text(size = 7))

### 2.5 PC1 vs PC3

In [ ]:
ggplot(data, aes(x = PC1, y = PC3, colour = Superpopulation.name)) +
  geom_point(alpha = 0.7, size = 1.5) +
  xlab("PC1") + ylab("PC3") +
  labs(colour = "Superpopulation", title = "PC1 vs PC3")

### 2.6 Within-European PCA

Running PCA only within Europeans reveals finer-scale structure: Northern Europeans (Finnish, British) tend to separate from Southern Europeans (Iberian, Tuscan).

In [ ]:
pc_cols_eur <- c("FID", "IID", paste0("PC", 1:10))
pca_eur     <- read.table("pca_EUR.eigenvec", header = FALSE, col.names = pc_cols_eur)
data_eur    <- merge(pca_eur, geo[, c("Sample.name", "Population.name")],
                     by.x = "IID", by.y = "Sample.name")

ggplot(data_eur, aes(x = PC1, y = PC2, colour = Population.name)) +
  geom_point(alpha = 0.8, size = 2) +
  xlab("PC1") + ylab("PC2") +
  labs(colour = "European population",
       title  = "PCA within European populations")

---
## Part III — Detecting Population Outliers

In a study designed to be homogeneous (e.g., European-only), individuals who cluster far from the main group in PC space likely have different ancestry. We flag them using a 3-standard-deviation rule around the EUR centroid.

In [ ]:
eur_mean_pc1 <- mean(data$PC1[data$Superpopulation.code == "EUR"])
eur_mean_pc2 <- mean(data$PC2[data$Superpopulation.code == "EUR"])
eur_sd_pc1   <- sd(data$PC1[data$Superpopulation.code == "EUR"])
eur_sd_pc2   <- sd(data$PC2[data$Superpopulation.code == "EUR"])

cat("EUR centroid: PC1 =", round(eur_mean_pc1, 4),
    ", PC2 =", round(eur_mean_pc2, 4), "\n")

data$eur_like <- abs(data$PC1 - eur_mean_pc1) < 3 * eur_sd_pc1 &
                 abs(data$PC2 - eur_mean_pc2) < 3 * eur_sd_pc2

cat("Individuals within 3 SD of EUR centroid:", sum(data$eur_like), "\n")

In [ ]:
ggplot(data, aes(x = PC1, y = PC2,
                 colour = Superpopulation.code,
                 shape  = eur_like)) +
  geom_point(alpha = 0.7, size = 1.5) +
  scale_shape_manual(values = c(4, 16),
                     labels = c("Excluded", "EUR-like (kept)")) +
  labs(colour = "Superpopulation", shape = "Selection",
       title  = "EUR-like individuals (within 3 SD of EUR centroid)")

In [ ]:
# Save sample list for PLINK
# Download from the Files panel (left sidebar → right-click → Download),
# upload to Cloud Shell and run:
#   plink --bfile hapmap3_qc --keep samples_EUR_like.txt --make-bed --out hapmap3_EUR
eur_keep <- data[data$eur_like, c("FID", "IID")]
write.table(eur_keep, "samples_EUR_like.txt",
            sep = " ", row.names = FALSE, col.names = FALSE, quote = FALSE)
cat("Saved", nrow(eur_keep), "EUR-like individuals to samples_EUR_like.txt\n")

---
## Bonus — Ancestry Prediction from PCA

PCA scores can predict the ancestry of individuals of unknown origin. This is the principle behind commercial genetic ancestry tests (23andMe, AncestryDNA, etc.).

We use **k-Nearest Neighbours (k-NN)**: for each individual, find the $k$ closest individuals in PC space among the labelled reference and assign the majority label.

In [ ]:
if (!requireNamespace("class", quietly = TRUE)) install.packages("class")
library(class)

# Features: top 10 PCs
pc_features <- paste0("PC", 1:10)
X <- as.matrix(data[, pc_features])
y <- data$Superpopulation.code

cat("Total individuals:", nrow(X), "\n")
print(table(y))

# 80/20 train/test split
set.seed(42)
n         <- nrow(data)
train_idx <- sample(n, size = floor(0.8 * n), replace = FALSE)
test_idx  <- setdiff(seq_len(n), train_idx)

train_X <- X[train_idx, ];  test_X <- X[test_idx, ]
train_y <- y[train_idx];    test_y <- y[test_idx]

cat("\nTraining set:", nrow(train_X), "individuals\n")
cat("Test set:    ", nrow(test_X),  "individuals\n")

In [ ]:
predicted <- knn(train = train_X, test = test_X, cl = train_y, k = 5)

conf_mat <- table(Predicted = predicted, True = test_y)
print(conf_mat)

accuracy <- sum(diag(conf_mat)) / sum(conf_mat)
cat("\nOverall accuracy:", round(accuracy * 100, 1), "%\n")

per_pop <- diag(conf_mat) / colSums(conf_mat)
cat("\nPer-population accuracy:\n")
print(round(per_pop * 100, 1))

In [ ]:
# Predict for all individuals and show misclassifications
predicted_all    <- knn(train = train_X, test = X, cl = train_y, k = 5)
data$predicted   <- as.character(predicted_all)
data$correct     <- data$predicted == data$Superpopulation.code

ggplot(data, aes(x = PC1, y = PC2,
                 colour = Superpopulation.code,
                 shape  = correct)) +
  geom_point(alpha = 0.7, size = 1.8) +
  scale_shape_manual(values = c(4, 16),
                     labels = c("Misclassified", "Correct")) +
  labs(colour = "True superpopulation", shape = "Classification",
       title  = "k-NN ancestry predictions (k = 5)")